In [ ]:
# RMW and R34 Distribution Fitting
# Author: Siméon Vareilles
# Description: Extracts RMW and R34 parameters from IBTrACS data and fits statistical distributions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pickle
import math
import statistics as stat
from tqdm import tqdm
from scipy.io import loadmat
from os.path import exists

# CONFIGURATION 

DATA_DIR = os.getenv("TC_DATA_DIR", "../data") # Note: points to parent data folder if inside 'analysis'
print(f"Reading data from: {DATA_DIR}")
MPI_DIR = os.path.join(TC_DATA_DIR, "mpi_out")

print(f"Loading data from: {TC_DATA_DIR}")

# To load data safely
def get_data_path(filename):
    path = os.path.join(TC_DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"Warning: {filename} not found at {path}")
    return path

In [38]:
plt.rcParams.update({
"text.usetex": True,
'font.size' : 20,
"font.family": "lmodern",
"ps.distiller.res": 10000})
plt.rcParams['figure.dpi'] = 250
plt.rcParams['savefig.dpi'] = 250
plt.style.use('seaborn-pastel')
plt.rcParams["figure.figsize"] = [5,4]
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18

In [ ]:
from global_land_mask import globe

def extract_params(datasets, param_name='RMW', method='last', filter_land=True):
    """
    Extracts RMW or R34 parameters.
    
    Arguments:
    param_name : 'RMW' or 'R34'
    method     : 'all' (all points), 'mean' (avg per storm), 'last' (last point)
    filter_land: If True, uses Lat/Lon to ignore points over land.
    """
    extracted_values = []
    
    print(f"Extracting {param_name} (Method: {method}, Filter Land: {filter_land})...")
    
    for ds in datasets:
        if param_name not in ds:
            continue
            
        # 1. Get the Data Arrays
        # We need wind/radius AND location to filter properly
        raw_data = ds[param_name].values
        lats = ds['lat'].values
        lons = ds['lon'].values
        
        # 2. Iterate through each storm
        
        for i in tqdm(range(len(raw_data)), desc=f"Processing {param_name} in Basin"):
            
            storm_track = raw_data[i]
            storm_lats = lats[i]
            storm_lons = lons[i]
            
            # 3. Create a Validity Mask
            # Must be: Finite (not NaN), Physical (< 10000), Positive
            valid_mask = np.isfinite(storm_track) & (storm_track < 10000) & (storm_track > 0)
            
            # 4. OPTIONAL: Filter out Land Points
            if filter_land:
                # global_land_mask requires inputs to be wrapped in checks to avoid errors with NaNs
                # We only check points that are already valid numbers
                # Note: globe.is_land returns True for Land, so we want ~ (Not Land)
                
                # Check lat/lon only where data is valid to save time
                is_land = globe.is_land(storm_lats, storm_lons)
                
                # Update mask: Valid AND (Not Land)
                valid_mask = valid_mask & (~is_land)

            # Apply mask
            valid_data = storm_track[valid_mask]
            
            if len(valid_data) == 0:
                continue

            # 5. Extract based on method
            if method == 'all':
                extracted_values.extend(valid_data)
            elif method == 'mean':
                extracted_values.append(np.mean(valid_data))
            elif method == 'last':
                extracted_values.append(valid_data[-1])

    return np.array(extracted_values)

In [ ]:
# EXECUTION 
# scale_factor = 1.852 (Nautical Miles to Km)

# Replicating your original count (~500 points):
rmw_data = extract_params(datasets, 'RMW', method='last') * 1.852 
r34_data = extract_params(datasets, 'R34', method='last') * 1.852

print(f"Extracted {len(rmw_data)} RMW values.")
print(f"Extracted {len(r34_data)} R34 values.")

In [ ]:
# Loading the observed data
basin_names = ["WP", "SP", "SI", "SA", "NI", "NA", "EP"]

data = []
for basin in basin_names:
    filename = f"IBTrACS_TRMM_SHIPS_{basin}_LFR.nc"
    try:
        ds = xr.open_dataset(get_data_path(filename))
        data.append(ds)
    except FileNotFoundError:
        print(f"Skipping {basin}: File not found.")

# Load topography
topography = loadmat(get_data_path("data_grd.mat"))

longi=np.array(topography["x"])[0]
#longi.flatten()
lati=np.array(topography["y"])[0]
# lati.flatten()
heights=np.array(topography["Z"])  # table of heights len(lati)=10801*len(longi)=21601

In [ ]:
basins = ["WP", "SP", "SI", "SA", "NI", "NA", "EP"]
datasets = []

print("Loading IBTrACS NetCDF files...")
for basin in basins:
    file_path = os.path.join(DATA_DIR, f"IBTrACS_TRMM_SHIPS_{basin}_LFR.nc")
    if os.path.exists(file_path):
        try:
            datasets.append(xr.open_dataset(file_path))
        except Exception as e:
            print(f"Error loading {basin}: {e}")

if not datasets:
    print("No datasets loaded. Check your DATA_DIR.")
else:
    # Extract RMW & R34 (Converted from nm to km: * 1.852)
    # Using method='last' to match your original legacy code
    rmw_data = extract_params(datasets, 'RMW', method='last', filter_land=True) * 1.852
    r34_data = extract_params(datasets, 'R34', method='last', filter_land=True) * 1.852

    print(f"\nFinal Count - RMW: {len(rmw_data)}, R34: {len(r34_data)}")


In [ ]:
#Fit & Plot RMW (Radius of Maximum Winds)
if len(rmw_data) > 0:
        print("\n--- Fitting Distributions for RMW ---")
        f_rmw = Fitter(rmw_data, distributions=['gamma', 'lognorm', 'beta', 'burr'])
        f_rmw.fit()
        print(f_rmw.summary())
        
        plt.figure(figsize=(10, 6))
        plt.hist(rmw_data, bins=50, density=True, alpha=0.6, color='skyblue', label='Observed RMW')
        plt.title("RMW Distribution Fit")
        plt.xlabel("Radius of Maximum Winds (km)")
        plt.ylabel("Probability Density")
        plt.legend()
        plt.savefig(os.path.join(DATA_DIR, "RMW_distribution_fit.png"))
        plt.show()

In [ ]:
#Fit & Plot R34 (Radius of 34kt Winds)
    if len(r34_data) > 0:
        print("\n--- Fitting Distributions for R34 ---")
        f_r34 = Fitter(r34_data, distributions=['gamma', 'lognorm', 'beta', 'burr'])
        f_r34.fit()
        print(f_r34.summary())
        
        plt.figure(figsize=(10, 6))
        plt.hist(r34_data, bins=50, density=True, alpha=0.6, color='salmon', label='Observed R34')
        plt.title("R34 Distribution Fit")
        plt.xlabel("Radius of 34kt Winds (km)")
        plt.ylabel("Probability Density")
        plt.legend()
        plt.savefig(os.path.join(DATA_DIR, "R34_distribution_fit.png"))
        plt.show()